# Week 3: Cleaning and Transforming Data

**Learning objective:** Diagnose data-quality issues, make justified transformations, and document checks.

**Expected outputs:** quality profile, cleaned teaching copy, derived variable, group summary, and transformation log.

**Data source:** Synthetic course data. The raw file is complete; this notebook deliberately creates a messy copy for practice. [Open in Colab](https://colab.research.google.com/github/GSIS-DS/data-science-ai-global-decision-making/blob/main/notebooks/week-03/03_cleaning_transforming_data.ipynb).

In [ ]:
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/GSIS-DS/data-science-ai-global-decision-making/main/data/sample/global_indicators_sample.csv"
LOCAL_PATHS = [
    Path("data/sample/global_indicators_sample.csv"),
    Path("../../data/sample/global_indicators_sample.csv"),
]
local_path = next((path for path in LOCAL_PATHS if path.exists()), None)
data_source = str(local_path) if local_path else DATA_URL
df = pd.read_csv(data_source)
print(f"Loaded {len(df)} rows from {data_source}")
df.head()

## 1. Inspect before changing

Never clean silently. Record row count, types, duplicates, and missing values before deciding what needs correction.

In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())

## 2. Practice on a deliberately messy copy

The next cell introduces one missing value, one duplicate row, and a year stored as text. These issues are instructional and are not present in the source CSV.

In [ ]:
messy = df.copy()
messy.loc[0, 'inflation'] = pd.NA
messy['year'] = messy['year'].astype(str)
messy = pd.concat([messy, messy.iloc[[1]]], ignore_index=True)

print('Missing inflation:', messy['inflation'].isna().sum())
print('Duplicate rows:', messy.duplicated().sum())
print('Year type:', messy['year'].dtype)

In [ ]:
clean = messy.drop_duplicates().copy()
clean['year'] = pd.to_numeric(clean['year'], errors='raise')

# Keep the missing value rather than inventing an inflation rate.
# Drop it only from calculations that require inflation.
clean['trade_balance_usd_bn'] = clean['exports_usd_bn'] - clean['imports_usd_bn']

assert clean.duplicated().sum() == 0
assert pd.api.types.is_numeric_dtype(clean['year'])
clean[['country', 'year', 'inflation', 'trade_balance_usd_bn']].head()

In [ ]:
regional_summary = (clean.dropna(subset=['inflation'])
                    .groupby(['region', 'year'], as_index=False)
                    .agg(mean_inflation=('inflation', 'mean'),
                         mean_trade_balance=('trade_balance_usd_bn', 'mean'))
                    .round(2))
regional_summary.head(10)

## Data-quality checklist and exercise

- Are required columns present and correctly typed?
- Are duplicates defined using the right keys?
- Is missingness handled without inventing evidence?
- Are units and transformations documented?

Write a 3-5 line transformation log using: **input issue -> change -> reason -> verification -> interpretive effect**.

**Example:** `year` arrived as text in the practice copy. I converted it with `pd.to_numeric(..., errors='raise')` so numeric comparisons are reliable. I verified the resulting dtype. This change does not alter values but makes invalid text fail visibly.